# Bronze: NOTAMs from Kafka
- **Source**: Event Hubs topic `notams-raw`
- **Target**: Delta table at `abfss://bronze@.../notams`

## Params

In [0]:
dbutils.widgets.text("mode", "backfill", "backfill | incremental")
dbutils.widgets.text("starting_offset", "earliest", "earliest | latest")

MODE = dbutils.widgets.get("mode")
STARTING_OFFSET = dbutils.widgets.get("starting_offset")

## Imports and Configs

In [0]:
email = dbutils.secrets.get(scope="flight-delay", key="db-email")

In [0]:
import sys
sys.path.insert(0, f"/Workspace/Users/{email}/ml-final-project/databricks")

from common.config import (
    BRONZE_NOTAMS_PATH,
    CHECKPOINT_NOTAMS,
    TOPIC_NOTAMS,
    get_kafka_options,
    get_secret,
    configure_storage_auth,
)

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

## Authenticate with ADLS Storage

In [0]:
configure_storage_auth(spark)

## Read Data from Event Hub - Kafka

In [0]:
eh_connection_string = get_secret("eventhubs-consumer-connection-string")

kafka_options = get_kafka_options(
    topic=TOPIC_NOTAMS,
    connection_string=eh_connection_string,
    starting_offset=STARTING_OFFSET,
)

raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)


## Bronzr Layer Minimum Transformations

In [0]:
bronze_df = (
    raw_stream
    .select(
        F.col("key").cast(StringType()).alias("kafka_key"),
        F.col("value").cast(StringType()).alias("kafka_value"),
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.current_timestamp().alias("ingestion_ts_utc"),
        F.lit(MODE).alias("ingestion_mode"),
    )
)


## Write to Delta Table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_NOTAMS)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start(BRONZE_NOTAMS_PATH)
)

query.awaitTermination()

### Register as Delta Table in UC

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.notams
    USING DELTA
    LOCATION '{BRONZE_NOTAMS_PATH}'
""")

### Test the Delta Table

In [0]:
row_count = spark.read.format("delta").load(BRONZE_NOTAMS_PATH).count()
print(f"Total rows in bronze.notams: {row_count:,}")

## Exit

In [0]:
dbutils.notebook.exit(f'{{"rows_in_bronze": {row_count}}}')